# 🚗 Cars24 Used Cars — Data Cleaning

This notebook cleans the raw Cars24 dataset while preserving the original raw file.

**Source:** `Cars24_Raw_data.csv`

**Goal:** Create a reliable `cars24_cleaned.csv` for EDA, area-wise analysis and brand-wise analysis.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

## 2. Load Raw Dataset

In [ ]:
df = pd.read_csv("Cars24_Raw_data.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

## 3. Understand the Dataset

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

## 4. Missing Values Check

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing

## 5. Duplicate Check

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate listing URLs:", df["listing_url"].duplicated().sum())

## 6. Standardize Text Columns

In [ ]:
text_cols = ["car_name", "brand", "model", "body_type", "fuel_type", "transmission", "price_currency", "source_website", "City"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Standardize common city naming
df["City"] = df["City"].replace({
    "Bengaluru": "Bangalore",
    "Gurugram": "Gurgaon"
})

# Standardize common brand casing
df["brand"] = df["brand"].str.title()

df.head()

## 7. Convert Numeric Columns

In [ ]:
numeric_cols = ["manufacturing_year", "price_numeric", "km_driven"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols].dtypes)

## 8. Validate Manufacturing Year

In [ ]:
current_year = pd.Timestamp.today().year

invalid_year = (
    df["manufacturing_year"].isna() |
    (df["manufacturing_year"] < 1990) |
    (df["manufacturing_year"] > current_year)
)

print("Invalid manufacturing years:", invalid_year.sum())

# Keep only valid years
df = df.loc[~invalid_year].copy()

## 9. Validate Price and Kilometres

In [ ]:
invalid_price = df["price_numeric"].isna() | (df["price_numeric"] <= 0)
invalid_km = df["km_driven"].isna() | (df["km_driven"] < 0)

print("Invalid prices:", invalid_price.sum())
print("Invalid KM values:", invalid_km.sum())

# Remove invalid records only
df = df.loc[~invalid_price & ~invalid_km].copy()

## 10. Validate Categorical Fields

In [ ]:
for col in ["brand", "body_type", "fuel_type", "transmission", "City"]:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head(20))

## 11. Create Useful Features

In [ ]:
df["Car_Age"] = current_year - df["manufacturing_year"]

df["Price_Lakh"] = df["price_numeric"] / 100000

# Useful for analysis later
df["KM_in_Thousands"] = df["km_driven"] / 1000

df.head()

## 12. Final Quality Check

In [ ]:
print("Final shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate URLs:", df["listing_url"].duplicated().sum())

## 13. Final Clean Dataset Preview

In [ ]:
display(df.head(10))
display(df.describe(include="all").T)

## 14. Save Cleaned Dataset

The raw file is not overwritten.

In [ ]:
df.to_csv("cars24_cleaned.csv", index=False)
print("✅ Saved: cars24_cleaned.csv")

## 15. Cleaning Summary

- Raw dataset preserved
- Missing values checked
- Duplicate rows checked
- Duplicate listing URLs checked
- Text fields standardized
- Numeric fields converted
- Manufacturing year validated
- Price and KM values validated
- City values standardized
- Car Age and Price in Lakh created
- Clean dataset exported for EDA